# SmartRoute-OSM: Benchmark Comparativo e Visualização em Mapa Interativo

Neste notebook final do pipeline, carregamos os resultados salvos das etapas anteriores para realizar uma análise comparativa do desempenho entre **Dijkstra** e **A***. Por fim, renderizamos a rota calculada sobre o mapa interativo com `Folium`.

In [9]:
import os
import json
import osmnx as ox
import pandas as pd
import folium

# Caminhos dos arquivos de métricas
data_dir = "../data"
dijkstra_csv = os.path.join(data_dir, "dijkstra_metrics.csv")
astar_csv = os.path.join(data_dir, "astar_metrics.csv")
path_json = os.path.join(data_dir, "astar_path.json")
graphml_path = os.path.join(data_dir, "quixada_drive.graphml")

# Verificar existência dos arquivos
for file_path in [dijkstra_csv, astar_csv, path_json, graphml_path]:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Arquivo necessário não encontrado: {file_path}. Execute os notebooks 01, 02 e 03 primeiro.")

print("Todos os arquivos necessários foram localizados!")

Todos os arquivos necessários foram localizados!


## 1. Tabela Comparativa de Desempenho
Unimos as estatísticas coletadas nas execuções anteriores para analisar a eficiência computacional de cada algoritmo.

In [10]:
df_dijkstra = pd.read_csv(dijkstra_csv)
df_astar = pd.read_csv(astar_csv)

# Unir os DataFrames
df_benchmark = pd.concat([df_dijkstra, df_astar], ignore_index=True)

# Calcular ganho de eficiência em porcentagem
nodes_dijkstra = df_dijkstra['visited_nodes'].iloc[0]
nodes_astar = df_astar['visited_nodes'].iloc[0]
reduction_pct = ((nodes_dijkstra - nodes_astar) / nodes_dijkstra) * 100

print("=== BENCHMARK COMPARATIVO ===")
print(df_benchmark[['algorithm', 'distance_m', 'visited_nodes', 'execution_time_ms']])
print(f"\nO algoritmo A* reduziu a exploração de nós em {reduction_pct:.2f}% em relação ao Dijkstra.")

=== BENCHMARK COMPARATIVO ===
        algorithm   distance_m  visited_nodes  execution_time_ms
0        Dijkstra  1890.272383            961           8.170128
1  A* (Haversine)  1890.272383             81           5.640507

O algoritmo A* reduziu a exploração de nós em 91.57% em relação ao Dijkstra.


## 2. Carregamento do Grafo e do Caminho
Carregamos a malha viária e a lista de nós da rota ideal para renderização espacial.

In [11]:
G = ox.load_graphml(graphml_path)

with open(path_json, "r") as f:
    route_path = json.load(f)

origem_node = route_path[0]
destino_node = route_path[-1]

origem_coords = (G.nodes[origem_node]['y'], G.nodes[origem_node]['x'])
destino_coords = (G.nodes[destino_node]['y'], G.nodes[destino_node]['x'])

print(f"Origem carregada: {origem_coords}")
print(f"Destino carregado: {destino_coords}")

Origem carregada: (-4.968511, -39.0161952)
Destino carregado: (-4.9801058, -39.0077824)


## 3. Visualização em Mapa Interativo (Folium)
Desenhamos o trajeto sobre o mapa com marcações nos pontos de início e término da navegação.

In [12]:
import folium

route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in route_path]

center_lat, center_lon = route_coords[0]
route_map = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="OpenStreetMap")

folium.PolyLine(
    locations=route_coords,
    color="#1f77b4",
    weight=6,
    opacity=0.8,
    popup="Rota Calculada"
).add_to(route_map)

folium.Marker(
    location=origem_coords,
    popup="<b>Origem</b>",
    icon=folium.Icon(color="green", icon="play", prefix="fa")
).add_to(route_map)

folium.Marker(
    location=destino_coords,
    popup="<b>Destino</b>",
    icon=folium.Icon(color="red", icon="flag", prefix="fa")
).add_to(route_map)

route_map.save(os.path.join(data_dir, "route_map.html"))
print("Mapa salvo em '../data/route_map.html' com sucesso!")

route_map

Mapa salvo em '../data/route_map.html' com sucesso!
